In [1]:
import pandas as pd

In [2]:
ssim_df = pd.read_csv('ssim_table_with_styles.csv')
ssim_df

,category,content,exp1,exp2,exp3,exp4,exp5,name1,name2,name3,name4,name5
0,landscape,beaulieu,0.209986,0.054180,0.022443,0.153979,0.169131,K,K,K,K,K
1,landscape,beaulieu,0.214029,0.080033,0.025743,0.157171,0.334289,V,V,V,V,M
2,landscape,beaulieu,0.303134,0.172214,0.028252,0.266713,0.335573,P,P,P,P,P
3,still-life,skull,0.101914,0.051823,0.015587,0.091950,0.429772,K,K,K,K,P
4,still-life,skull,0.305680,0.189936,0.020415,0.297561,0.344851,M,M,M,M,M
5,still-life,skull,0.316816,0.195207,0.023217,0.297387,0.343552,T,T,T,T,T
6,portrait,girl,0.415137,0.260297,0.034836,0.411860,0.462338,P,P,P,P,P
7,portrait,girl,0.295497,0.136942,0.031002,0.328436,0.339268,T,T,T,T,T
8,portrait,girl,0.230094,0.102864,0.029521,0.216930,0.219434,V,V,V,V,V
9,portrait,man,0.235351,0.102899,0.027693,0.179743,0.241078,V,V,V,V,V


In [3]:
import pandas as pd

parts = []

for i in range(1, 6):
    tmp = ssim_df[["category", "content", f"name{i}", f"exp{i}"]].copy()

    tmp = tmp.rename(columns={
        f"name{i}": "style",
        f"exp{i}": "value"
    })

    tmp["experiment"] = f"exp{i}"
    parts.append(tmp)

df_long = pd.concat(parts, ignore_index=True)
df_long

,category,content,style,value,experiment
0,landscape,beaulieu,K,0.209986,exp1
1,landscape,beaulieu,V,0.214029,exp1
2,landscape,beaulieu,P,0.303134,exp1
3,still-life,skull,K,0.101914,exp1
4,still-life,skull,M,0.305680,exp1
...,...,...,...,...,...
130,portrait,repin,T,0.445047,exp5
131,portrait,repin,P,0.508120,exp5
132,still-life,fish,T,0.230553,exp5
133,still-life,fish,M,0.220414,exp5


In [4]:
df_long["normalized_value"] = (df_long["value"] / df_long.groupby("experiment")["value"].transform("mean") - 1)
df_long

,category,content,style,value,experiment,normalized_value
0,landscape,beaulieu,K,0.209986,exp1,-0.248960
1,landscape,beaulieu,V,0.214029,exp1,-0.234500
2,landscape,beaulieu,P,0.303134,exp1,0.084196
3,still-life,skull,K,0.101914,exp1,-0.635492
4,still-life,skull,M,0.305680,exp1,0.093303
...,...,...,...,...,...,...
130,portrait,repin,T,0.445047,exp5,0.262534
131,portrait,repin,P,0.508120,exp5,0.441463
132,still-life,fish,T,0.230553,exp5,-0.345955
133,still-life,fish,M,0.220414,exp5,-0.374719


In [5]:
df_long.groupby('category')['normalized_value'].describe()

,count,mean,std,min,25%,50%,75%,max
category,,,,,,,,
landscape,45.0,-0.083616,0.463978,-0.732846,-0.417608,-0.175944,0.166262,1.284404
portrait,45.0,0.151298,0.333804,-0.377497,-0.086174,0.138285,0.440785,0.868058
still-life,45.0,-0.067682,0.392844,-0.655024,-0.374719,-0.121102,0.212286,1.153592


In [6]:
import statsmodels.formula.api as smf

In [7]:
model_no_category = smf.mixedlm(
    "normalized_value ~ C(experiment) + C(style)",
    data=df_long,
    groups=df_long["content"]
).fit(reml=False)
model_no_category.summary()

<class 'statsmodels.iolib.summary2.Summary'>
"""
             Mixed Linear Model Regression Results
================================================================
Model:              MixedLM Dependent Variable: normalized_value
No. Observations:   135     Method:             ML              
No. Groups:         9       Scale:              0.0453          
Min. group size:    15      Log-Likelihood:     8.2548          
Max. group size:    15      Converged:          Yes             
Mean group size:    15.0                                        
----------------------------------------------------------------
                      Coef.  Std.Err.   z    P>|z| [0.025 0.975]
----------------------------------------------------------------
Intercept             -0.428    0.073 -5.859 0.000 -0.572 -0.285
C(experiment)[T.exp2] -0.000    0.058 -0.000 1.000 -0.114  0.114
C(experiment)[T.exp3] -0.000    0.058 -0.000 1.000 -0.114  0.114
C(experiment)[T.exp4] -0.000    0.058 -0.000 1.000 -0.114  0.114
C(experiment)[T.exp5] -0.150    0.059 -2.545 0.011 -0.265 -0.034
C(style)[T.M]          0.679    0.068  9.916 0.000  0.545  0.813
C(style)[T.P]          0.859    0.060 14.440 0.000  0.743  0.976
C(style)[T.T]          0.537    0.062  8.718 0.000  0.417  0.658
C(style)[T.V]          0.192    0.064  3.016 0.003  0.067  0.316
Group Var              0.020    0.053                           
================================================================

"""

In [8]:
model_with_category = smf.mixedlm(
    "normalized_value ~ C(category) + C(experiment) + C(style)",
    data=df_long,
    groups=df_long["content"]
).fit(reml=False)
model_with_category.summary()

<class 'statsmodels.iolib.summary2.Summary'>
"""
               Mixed Linear Model Regression Results
====================================================================
Model:              MixedLM   Dependent Variable:   normalized_value
No. Observations:   135       Method:               ML              
No. Groups:         9         Scale:                0.0453          
Min. group size:    15        Log-Likelihood:       9.1547          
Max. group size:    15        Converged:            Yes             
Mean group size:    15.0                                            
--------------------------------------------------------------------
                          Coef.  Std.Err.   z    P>|z| [0.025 0.975]
--------------------------------------------------------------------
Intercept                 -0.429    0.094 -4.570 0.000 -0.612 -0.245
C(category)[T.portrait]    0.092    0.114  0.807 0.420 -0.131  0.315
C(category)[T.still-life] -0.071    0.116 -0.612 0.541 -0.297  0.156
C(experiment)[T.exp2]     -0.000    0.058 -0.000 1.000 -0.113  0.113
C(experiment)[T.exp3]     -0.000    0.058 -0.000 1.000 -0.113  0.113
C(experiment)[T.exp4]     -0.000    0.058 -0.000 1.000 -0.113  0.113
C(experiment)[T.exp5]     -0.151    0.059 -2.564 0.010 -0.266 -0.035
C(style)[T.M]              0.685    0.069  9.931 0.000  0.550  0.820
C(style)[T.P]              0.847    0.060 14.123 0.000  0.730  0.965
C(style)[T.T]              0.531    0.062  8.501 0.000  0.408  0.653
C(style)[T.V]              0.176    0.064  2.744 0.006  0.050  0.302
Group Var                  0.016    0.043                           
====================================================================

"""

In [9]:
from scipy.stats import chi2

lr = 2 * (model_with_category.llf - model_no_category.llf)
df_diff = (model_with_category.df_modelwc - model_no_category.df_modelwc)

p = chi2.sf(lr, df_diff)

print("p-value:", p)

p-value: 0.406612889999304


In [10]:
import statsmodels.formula.api as smf

model_base = smf.mixedlm("normalized_value ~ C(category) + C(experiment) + C(style)",
                         data=df_long,
                         groups=df_long["content"])

result_base = model_base.fit(reml=False)

print(result_base.summary())

               Mixed Linear Model Regression Results
Model:              MixedLM   Dependent Variable:   normalized_value
No. Observations:   135       Method:               ML              
No. Groups:         9         Scale:                0.0453          
Min. group size:    15        Log-Likelihood:       9.1547          
Max. group size:    15        Converged:            Yes             
Mean group size:    15.0                                            
--------------------------------------------------------------------
                          Coef.  Std.Err.   z    P>|z| [0.025 0.975]
--------------------------------------------------------------------
Intercept                 -0.429    0.094 -4.570 0.000 -0.612 -0.245
C(category)[T.portrait]    0.092    0.114  0.807 0.420 -0.131  0.315
C(category)[T.still-life] -0.071    0.116 -0.612 0.541 -0.297  0.156
C(experiment)[T.exp2]     -0.000    0.058 -0.000 1.000 -0.113  0.113
C(experiment)[T.exp3]     -0.000    0.058 -0.000 1

In [11]:
model_interaction = smf.mixedlm(
    "normalized_value ~ C(category) * C(experiment) + C(style)",
    data=df_long,
    groups=df_long["content"])

result_interaction = model_interaction.fit(reml=False)

print(result_interaction.summary())

                          Mixed Linear Model Regression Results
Model:                      MixedLM          Dependent Variable:          normalized_value
No. Observations:           135              Method:                      ML              
No. Groups:                 9                Scale:                       0.0431          
Min. group size:            15               Log-Likelihood:              12.1621         
Max. group size:            15               Converged:                   Yes             
Mean group size:            15.0                                                          
------------------------------------------------------------------------------------------
                                                Coef.  Std.Err.   z    P>|z| [0.025 0.975]
------------------------------------------------------------------------------------------
Intercept                                       -0.481    0.107 -4.488 0.000 -0.691 -0.271
C(category)[T.portrait]   

In [12]:
from scipy.stats import chi2

lr = 2 * (result_interaction.llf - result_base.llf)
df_diff = (result_interaction.df_modelwc - result_base.df_modelwc)
p_value = chi2.sf(lr, df_diff)

print(lr)
print(df_diff)
print("p-value:", p_value)

6.014870962455802
8
p-value: 0.645566035237427


In [13]:
model_no_style = smf.mixedlm(
    "normalized_value ~ C(category) + C(experiment)",
    data=df_long,
    groups=df_long["content"]
)

result_no_style = model_no_style.fit(
    reml=False,
    method="lbfgs"
)

print(result_no_style.converged)

/opt/homebrew/anaconda3/lib/python3.12/site-packages/statsmodels/base/model.py:589: SingularMatrixWarning: Random effects covariance is singular
  return -self.score(params, *args) / nobs


True


/var/folders/_2/nrxg9rv974ddvc3x_62_jj1w0000gn/T/ipykernel_29389/3156962138.py:7: ConvergenceWarning: The MLE may be on the boundary of the parameter space.
  result_no_style = model_no_style.fit(


In [14]:
from scipy.stats import chi2

lr = 2 * (result_base.llf - result_no_style.llf)

df_diff = (result_base.df_modelwc - result_no_style.df_modelwc)

p = chi2.sf(lr, df_diff)

print(lr)
print(df_diff)
print("p-value:", p)

149.17129111359657
4
p-value: 3.064114797081872e-31


In [15]:
print(result_no_style.summary())

               Mixed Linear Model Regression Results
Model:              MixedLM   Dependent Variable:   normalized_value
No. Observations:   135       Method:               ML              
No. Groups:         9         Scale:                0.1479          
Min. group size:    15        Log-Likelihood:       -65.4309        
Max. group size:    15        Converged:            Yes             
Mean group size:    15.0                                            
--------------------------------------------------------------------
                          Coef.  Std.Err.   z    P>|z| [0.025 0.975]
--------------------------------------------------------------------
Intercept                 -0.084    0.103 -0.811 0.418 -0.286  0.119
C(category)[T.portrait]    0.235    0.112  2.100 0.036  0.016  0.454
C(category)[T.still-life]  0.016    0.112  0.142 0.887 -0.203  0.235
C(experiment)[T.exp2]     -0.000    0.105 -0.000 1.000 -0.205  0.205
C(experiment)[T.exp3]      0.000    0.105  0.000 1